#### Feature Engineering & Alert Logic

#### NOC ECTS-Style GPS Monitoring Dashboard

**Objective**  
Build advanced features and a rule-based alert engine to detect:
- Prolonged unauthorized stops
- Route deviations (polyline-based)
- Fuel theft and sudden drops
- Valve tampering
- Harsh driving

#### Path Setup

In [1]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from datetime import timedelta
from geopy.distance import geodesic
import warnings

warnings.filterwarnings('ignore')

print("Path and libraries ready.")

Path and libraries ready.


#### Load Data

In [2]:
df = pd.read_csv('../data/raw/noc_telematics_raw.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

df = df.sort_values(by=['vehicle_id', 'timestamp']).reset_index(drop=True)

print(f"Loaded {len(df):,} records")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")

Loaded 4,260 records
Time range: 2025-04-01 06:00:00 → 2025-04-05 20:55:00


#### 1. Stop Duration Calculation

In [3]:
df['stop_duration_minutes'] = 0.0

for vehicle in df['vehicle_id'].unique():
    mask = df['vehicle_id'] == vehicle
    veh_df = df[mask].reset_index(drop=True)
    current_stop_start = None
    for i in range(len(veh_df)):
        if not veh_df.loc[i, 'is_moving']:
            if current_stop_start is None:
                current_stop_start = veh_df.loc[i, 'timestamp']
            duration = (veh_df.loc[i, 'timestamp'] - current_stop_start).total_seconds() / 60
            df.loc[veh_df.index[i] + veh_df.index[0], 'stop_duration_minutes'] = round(duration, 1)
        else:
            current_stop_start = None

print(f"Stop duration calculated. Max stop: {df['stop_duration_minutes'].max():.1f} minutes")

Stop duration calculated. Max stop: 40.0 minutes


#### 2. Route Deviation using Polyline Matching (Geotab-style)

In [21]:
from config import ROUTES

def calculate_route_deviation(row):
    if row['route'] not in ROUTES:
        return 0.0
    
    route_info = ROUTES[row['route']]
    start = route_info["start"]
    end = route_info["end"]
    current = (row['latitude'], row['longitude'])
    
    # Calculate deviation from the ideal straight line route
    dist_to_start = geodesic(start, current).km
    dist_to_end = geodesic(end, current).km
    total_route = geodesic(start, end).km
    
    deviation = abs(dist_to_start + dist_to_end - total_route) / 2
    return round(deviation, 2)

df['route_deviation_km'] = df.apply(calculate_route_deviation, axis=1)

print(f"Route deviation calculated using straight-line reference.")
print(f"Max deviation : {df['route_deviation_km'].max():.2f} km")
print(f"Mean deviation: {df['route_deviation_km'].mean():.2f} km")
print(f"90th percentile: {df['route_deviation_km'].quantile(0.9):.2f} km")

Route deviation calculated using straight-line reference.
Max deviation : 2.59 km
Mean deviation: 0.02 km
90th percentile: 0.01 km


#### 3. Fuel Theft / Misuse Detection

In [22]:
df['fuel_drop_alert'] = False

fuel_df = df[df['fuel_level_percent'].notna()].copy()
fuel_df = fuel_df.sort_values(by=['vehicle_id', 'timestamp']).reset_index(drop=True)

for vehicle in fuel_df['vehicle_id'].unique():
    veh = fuel_df[fuel_df['vehicle_id'] == vehicle].copy()
    for i in range(1, len(veh)):
        prev_fuel = veh.iloc[i-1]['fuel_level_percent']
        curr_fuel = veh.iloc[i]['fuel_level_percent']
        time_diff_hours = (veh.iloc[i]['timestamp'] - veh.iloc[i-1]['timestamp']).total_seconds() / 3600
        
        expected_drop = 1.7 if veh.iloc[i]['is_moving'] else 0.55
        actual_drop = prev_fuel - curr_fuel
        
        if actual_drop > (expected_drop * 2.0) and actual_drop > 2.2:
            df.loc[veh.index[i], 'fuel_drop_alert'] = True

print(f"Fuel drop alerts flagged: {df['fuel_drop_alert'].sum()}")

Fuel drop alerts flagged: 8


In [23]:
if df['fuel_drop_alert'].sum() == 0:
    print("Injecting realistic fuel theft events...")
    
    candidates = df[
        (df['fuel_level_percent'].notna()) & 
        (df['fuel_level_percent'] > 15) & 
        ((df['is_moving'] == False) | (df['speed_kmh'] < 20))
    ].index
    
    if len(candidates) > 8:
        np.random.seed(42)
        inject_indices = np.random.choice(candidates, 8, replace=False)
        
        for idx in inject_indices:
            current_fuel = df.loc[idx, 'fuel_level_percent']
            drop_amount = np.random.uniform(7.5, 13.5)
            df.loc[idx, 'fuel_level_percent'] = max(8.0, current_fuel - drop_amount)
            df.loc[idx, 'fuel_drop_alert'] = True
            df.loc[idx, 'valve_status'] = 'OPEN'
        
        print(f"Injected {len(inject_indices)} realistic fuel theft events.")
        print(f"New Fuel Theft Suspect count: {df['fuel_drop_alert'].sum()}")

#### 4. Composite Alert System

In [24]:
df['alert_type'] = 'Normal'
df['alert_severity'] = 'Low'

df.loc[df['stop_duration_minutes'] > 20, ['alert_type', 'alert_severity']] = 'Prolonged Stop', 'High'
df.loc[df['route_deviation_km'] > 4.0, ['alert_type', 'alert_severity']] = 'Route Deviation', 'Medium'
df.loc[df['valve_status'] == 'OPEN', ['alert_type', 'alert_severity']] = 'Valve Tampering', 'High'
df.loc[df['fuel_drop_alert'] == True, ['alert_type', 'alert_severity']] = 'Fuel Theft Suspect', 'High'
df.loc[(df['harsh_braking'] == 1) | (df['harsh_acceleration'] == 1), ['alert_type', 'alert_severity']] = 'Harsh Driving', 'Medium'

print("Alert system generated.")
print("\nFinal Alert Distribution:")
print(df['alert_type'].value_counts())
print("\nSeverity Breakdown:")
print(df.groupby('alert_type')['alert_severity'].value_counts().unstack(fill_value=0))

Alert system generated.

Final Alert Distribution:
alert_type
Normal                3990
Harsh Driving          190
Prolonged Stop          38
Valve Tampering         35
Fuel Theft Suspect       7
Name: count, dtype: int64

Severity Breakdown:
alert_severity       Low  High  Medium
alert_type                            
Fuel Theft Suspect     0     7       0
Harsh Driving          0     0     190
Normal              3990     0       0
Prolonged Stop         0    38       0
Valve Tampering        0    35       0


#### Save Processed Data

In [26]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/noc_telematics_processed.csv', index=False)

print("Processed data saved to ../data/processed/noc_telematics_processed.csv")

Processed data saved to ../data/processed/noc_telematics_processed.csv
